# DeBERTa Fine-Tuning

**Run this notebook on Google Colab with a GPU runtime (T4 is fine).**

The two previous models work on hand-crafted features. DeBERTa reads the raw response text directly and learns whatever quality signals live in the language itself. It's slower to train, harder to interpret, and usually better at picking up on things like coherence and factual consistency that no hand-crafted feature captures.

When training is done, download the checkpoint folder and place it at `models/deberta_checkpoint/` in your local repo. Notebook 03 will load it from there for evaluation.

---
**Colab setup:** Runtime > Change runtime type > T4 GPU

In [ ]:
# install dependencies on Colab
# comment this out if running locally with the project env already set up
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, roc_auc_score

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Load data

Two options depending on your setup:
- Option A: upload `hh_rlhf_long.parquet` to Colab (Files panel on the left)
- Option B: load HH-RLHF directly from HuggingFace and re-extract responses

Option A is faster if you already have the file from running notebook 01.

In [ ]:
# Option A -- load from uploaded parquet
df = pd.read_parquet('hh_rlhf_long.parquet')

# Option B -- load directly from HuggingFace (uncomment if needed)
# from datasets import load_dataset
# raw = load_dataset('anthropic/hh-rlhf', data_dir='helpful-base')
# ... extract responses as in loader.py

print(f'Loaded {len(df):,} rows')
df.head(3)

In [ ]:
# reproduce the same 70/15/15 split
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df_shuffled)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_df = df_shuffled.iloc[:n_train]
val_df   = df_shuffled.iloc[n_train : n_train + n_val]
test_df  = df_shuffled.iloc[n_train + n_val :]

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

## Tokenize

DeBERTa-v3-small has a max token length of 512. Most responses fit, but we truncate anything longer. Padding is handled dynamically per batch.

In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['response'],
        truncation=True,
        max_length=MAX_LEN,
        padding=False,  # dynamic padding via data collator
    )

def make_hf_dataset(df):
    ds = Dataset.from_dict({
        'response': df['response'].tolist(),
        'labels': df['label'].tolist(),
    })
    return ds.map(tokenize, batched=True, remove_columns=['response'])

print('Tokenizing...')
train_ds = make_hf_dataset(train_df)
val_ds   = make_hf_dataset(val_df)
test_ds  = make_hf_dataset(test_df)
print('Done.')
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## Fine-tune DeBERTa-v3-small

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    return {
        'f1': f1_score(labels, preds, average='macro'),
        'auc': roc_auc_score(labels, probs),
    }

training_args = TrainingArguments(
    output_dir='deberta_checkpoint',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    report_to='none',  # set to 'wandb' if you want W&B logging
)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
print('Starting training...')
trainer.train()
print('Training complete.')

## Evaluate on test set

In [ ]:
test_preds = trainer.predict(test_ds)

logits = test_preds.predictions
labels = test_preds.label_ids
preds  = np.argmax(logits, axis=1)
probs  = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

deberta_auc = roc_auc_score(labels, probs)
deberta_f1  = f1_score(labels, preds, average='macro')

print(f'DeBERTa-v3-small -- Test AUC: {deberta_auc:.4f}  F1 (macro): {deberta_f1:.4f}')

In [ ]:
# save test predictions alongside the model
deberta_preds_df = pd.DataFrame({
    'y_true': labels,
    'deberta_prob': probs,
    'deberta_pred': preds,
})
deberta_preds_df.to_parquet('deberta_test_predictions.parquet', index=False)
print('Saved deberta_test_predictions.parquet')

## Save checkpoint

Save the best model and tokenizer so we can load them locally.

In [ ]:
trainer.save_model('deberta_checkpoint')
tokenizer.save_pretrained('deberta_checkpoint')
print('Checkpoint saved to deberta_checkpoint/')

## Download instructions

In the Colab Files panel (left sidebar), right-click the `deberta_checkpoint` folder and download it. Also download `deberta_test_predictions.parquet`.

Place them at:
```
models/deberta_checkpoint/          (the checkpoint folder)
data/processed/deberta_test_predictions.parquet
```

Then continue with notebook 03 on your local machine.

In [ ]:
# optional: zip the checkpoint for easier download
import shutil
shutil.make_archive('deberta_checkpoint', 'zip', 'deberta_checkpoint')
print('Zipped to deberta_checkpoint.zip -- download this from the Files panel')